In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import f1_score
from torch_geometric.nn import GATConv

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

graph_path = ROOT / "data/processed/protein_function_graph_aac_score700_split.pt"

graph = torch.load(graph_path, weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph = graph.to(device)

print("Device:", device)
print(graph)

c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cuda
Data(x=[18841, 22], edge_index=[2, 1067636], y=[18841, 96], edge_weight=[1067636], labeled_mask=[18841], train_mask=[18841], validation_mask=[18841], test_mask=[18841])


In [2]:
class GATBaseline(torch.nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_heads=2,
        dropout=0.30,
    ):
        super().__init__()

        self.gat1 = GATConv(
            input_dim,
            hidden_dim,
            heads=num_heads,
            concat=True,
            dropout=dropout,
            edge_dim=1,
        )

        self.gat2 = GATConv(
            hidden_dim * num_heads,
            output_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=1,
        )

        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight):
        edge_attr = edge_weight.unsqueeze(-1)

        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.gat2(x, edge_index, edge_attr=edge_attr)

In [10]:
INPUT_DIM = graph.x.shape[1]
HIDDEN_DIM = 32
NUM_HEADS = 2
OUTPUT_DIM = graph.y.shape[1]
DROPOUT = 0.30
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4

model = GATBaseline(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
# Class imbalance handle karne ke liye training labels se weights
train_labels = graph.y[graph.train_mask]

positive_count = train_labels.sum(dim=0)
negative_count = train_labels.shape[0] - positive_count

pos_weight = negative_count / positive_count

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [11]:
@torch.no_grad()
def evaluate(mask, threshold=0.5):
    model.eval()

    logits = model(graph.x, graph.edge_index, graph.edge_weight)
    loss = criterion(logits[mask], graph.y[mask])

    probabilities = torch.sigmoid(logits[mask]).cpu().numpy()
    targets = graph.y[mask].cpu().numpy()
    predictions = (probabilities >= threshold).astype(int)

    return {
        "loss": float(loss.item()),
        "micro_f1": f1_score(
            targets,
            predictions,
            average="micro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

In [12]:
EPOCHS = 200
PATIENCE = 30

best_validation_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(graph.x, graph.edge_index, graph.edge_weight)

    train_loss = criterion(
        logits[graph.train_mask],
        graph.y[graph.train_mask],
    )

    train_loss.backward()
    optimizer.step()

    validation_metrics = evaluate(graph.validation_mask)

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(train_loss.item()),
            "validation_loss": validation_metrics["loss"],
            "validation_micro_f1": validation_metrics["micro_f1"],
            "validation_macro_f1": validation_metrics["macro_f1"],
        }
    )

    if validation_metrics["loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train loss: {train_loss.item():.4f} | "
            f"val loss: {validation_metrics['loss']:.4f} | "
            f"val micro-F1: {validation_metrics['micro_f1']:.4f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_validation_loss:.4f}")

Epoch 001 | train loss: 1.9382 | val loss: 1.3583 | val micro-F1: 0.0518
Epoch 010 | train loss: 1.2519 | val loss: 1.2346 | val micro-F1: 0.0879
Epoch 020 | train loss: 1.1362 | val loss: 1.1199 | val micro-F1: 0.1022
Epoch 030 | train loss: 1.0682 | val loss: 1.0682 | val micro-F1: 0.1201
Epoch 040 | train loss: 1.0299 | val loss: 1.0270 | val micro-F1: 0.1288
Epoch 050 | train loss: 1.0061 | val loss: 1.0025 | val micro-F1: 0.1334
Epoch 060 | train loss: 0.9844 | val loss: 0.9864 | val micro-F1: 0.1364
Epoch 070 | train loss: 0.9667 | val loss: 0.9732 | val micro-F1: 0.1392
Epoch 080 | train loss: 0.9571 | val loss: 0.9637 | val micro-F1: 0.1444
Epoch 090 | train loss: 0.9409 | val loss: 0.9564 | val micro-F1: 0.1436
Epoch 100 | train loss: 0.9336 | val loss: 0.9488 | val micro-F1: 0.1440
Epoch 110 | train loss: 0.9375 | val loss: 0.9345 | val micro-F1: 0.1454
Epoch 120 | train loss: 0.9289 | val loss: 0.9417 | val micro-F1: 0.1472
Epoch 130 | train loss: 0.9234 | val loss: 0.9339 |

In [13]:
# Validation set par best probability threshold choose karenge.
threshold_candidates = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in threshold_candidates:
    metrics = evaluate(graph.validation_mask, threshold=threshold)
    threshold_results.append(
        {
            "threshold": round(float(threshold), 2),
            "validation_micro_f1": metrics["micro_f1"],
            "validation_macro_f1": metrics["macro_f1"],
        }
    )

threshold_df = pd.DataFrame(threshold_results)

best_threshold_row = threshold_df.loc[
    threshold_df["validation_micro_f1"].idxmax()
]

best_threshold = float(best_threshold_row["threshold"])

print("Chosen threshold:", best_threshold)
threshold_df

Chosen threshold: 0.85


,threshold,validation_micro_f1,validation_macro_f1
0,0.10,0.063324,0.063947
1,0.15,0.070323,0.073271
2,0.20,0.077939,0.083640
3,0.25,0.086532,0.094514
4,0.30,0.096196,0.105775
5,0.35,0.107237,0.117796
6,0.40,0.119805,0.130353
7,0.45,0.135071,0.144063
8,0.50,0.151515,0.158559
9,0.55,0.170521,0.174084


In [14]:
train_metrics = evaluate(graph.train_mask, threshold=best_threshold)
validation_metrics = evaluate(graph.validation_mask, threshold=best_threshold)
test_metrics = evaluate(graph.test_mask, threshold=best_threshold)

results_df = pd.DataFrame(
    [
        {"split": "train", **train_metrics},
        {"split": "validation", **validation_metrics},
        {"split": "test", **test_metrics},
    ]
)

results_df

,split,loss,micro_f1,macro_f1
0,train,0.843913,0.291787,0.222465
1,validation,0.906531,0.284349,0.220109
2,test,0.887977,0.266505,0.198148


In [15]:
models_dir = ROOT / "models"
results_dir = ROOT / "results"

models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

model_path = models_dir / "gat_aac_score700.pt"
metrics_path = results_dir / "gat_aac_score700_metrics.csv"
history_path = results_dir / "gat_aac_score700_training_history.csv"
torch.save(
    {
        "model_state_dict": best_state,
        "input_dim": INPUT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "output_dim": OUTPUT_DIM,
        "dropout": DROPOUT,
        "num_heads": NUM_HEADS,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
    },
    model_path,
)

results_df.to_csv(metrics_path, index=False)
pd.DataFrame(history).to_csv(history_path, index=False)

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)
print("Saved training history:", history_path)

Saved model: c:\Users\rajdeep\Desktop\Capstone Project_GNN\models\gat_aac_score700.pt
Saved metrics: c:\Users\rajdeep\Desktop\Capstone Project_GNN\results\gat_aac_score700_metrics.csv
Saved training history: c:\Users\rajdeep\Desktop\Capstone Project_GNN\results\gat_aac_score700_training_history.csv
